# Figure 4 — resurrected carbonic anhydrases

Reproduces the panels of this figure. Each cell runs a released producer and shows its output.

**Default level is 1** — redraw from a table shipped in the Zenodo deposit. No model, no GPU;
seconds on a laptop. Cells that need a GPU or a checkpoint are marked and left commented out.

Run from the `peint-paper` repository, or set `PEINT_PAPER_REPO` to point at it.

In [ ]:
import os, sys, subprocess, pathlib
# Point these at your checkout. PAPER_REPO is the peint-paper repository; PEINT_REPO the model
# library. Everything below runs from the paper repository root.
PAPER_REPO = pathlib.Path(os.environ.get("PEINT_PAPER_REPO", "..")).resolve()
os.environ.setdefault("PEINT_PAPER_PEINT_REPO", str(PAPER_REPO.parent / "peint"))
os.environ["PEINT_PAPER_LOCAL_DATA_ONLY"] = "1"
os.chdir(PAPER_REPO)
sys.path.insert(0, str(PAPER_REPO))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# Inline display needs IPython. These notebooks are also executed headlessly in CI, where it is
# absent, so fall back to printing rather than failing the whole notebook on the import.
try:
    from IPython.display import display, Image
    _INLINE = True
except ImportError:
    _INLINE = False
    def display(x): print(x)
    def Image(filename=None, width=None): return f"[figure written to {filename}]"

def run(module, *args):
    """Run a panel producer and show its output."""
    cmd = [sys.executable, "-m", module, *args]
    print("$", " ".join(cmd[2:]))
    r = subprocess.run(cmd, capture_output=True, text=True)
    tail = [l for l in r.stdout.split("\n") if l.strip()][-15:]
    print("\n".join(tail))
    if r.returncode:
        print(f"\n  *** FAILED (exit {r.returncode}) ***")
        print("  " + r.stderr.strip().split("\n")[-1][:400])
    return r.returncode == 0

def show(*names, w=760):
    """Display panel images from figures/output."""
    for n in names:
        p = PAPER_REPO / "figures" / "output" / (n + ".png")
        if p.exists():
            display(Image(filename=str(p), width=w))
        elif (p.with_suffix(".pdf")).exists():
            print(f"  {n}.pdf written (no PNG to display inline)")
        else:
            print("  not found:", n)

def compare(label, ours, printed, tol=0.05):
    """Print our value against the manuscript's."""
    ok = "MATCH" if abs(ours - printed) <= tol else "CHECK"
    print(f"  {label:<28} ours {ours:>8.3f}   manuscript {printed:>8.3f}   {ok}")

## Where these panels come from

Figure 4 is the wet-lab figure and is the one part of the paper whose panels live in
notebooks already committed to the repository, under
`figures/carbonic_anhydrase_figure/`. Every input is in-repo, so these are **level 1**:
nothing to download.

| panel | notebook |
|---|---|
| 4a, 4b tree and structures | `computational/tree_analysis.ipynb` (needs `PyQt5`) |
| 4c pLDDT vs distance | `computational/tree_analysis.ipynb` |
| 4d growth curves | `in_vivo/Experimental.ipynb` (needs the `growth` parser) |
| 4e stopped-flow kinetics | `in_vitro/stopped_flow_analysis.ipynb` |
| 4f natural kcat distribution | `computational/Brenda_CA.ipynb` |

Open them directly, or execute them headlessly with the helper below.

In [ ]:
NB = PAPER_REPO / "figures" / "carbonic_anhydrase_figure"
def run_nb(rel, *extra):
    """Execute a committed notebook's code cells as a script."""
    ok = run("scripts.run_notebook", str(NB / rel), *extra)
    return ok

def show_ca(*rels, w=760):
    """Figure 4 panels are written beside their notebooks, not into figures/output."""
    for rel in rels:
        p = NB / rel
        if p.exists() and p.suffix == ".png": display(Image(filename=str(p), width=w))
        elif p.exists(): print(f"  wrote {rel}")
        else: print(f"  not found: {rel}")

## 4c — predicted pLDDT vs phylogenetic distance

`ete3` pulls in PyQt5 only for the tree *rendering* in 4a/4b. Stubbing those symbols lets
4c run without a display; the plot itself does not use them.

> One cell will report `TypeError: 'NodeStyle' object does not support item assignment`.
> That is the 4a/4b tree-drawing code hitting the stub, and it is expected — **4c is
> unaffected** and its PDF is written below. Install PyQt5 if you want 4a/4b too.

In [ ]:
run_nb("computational/tree_analysis.ipynb",
       "--stub", "ete3:TreeStyle,ete3:NodeStyle,ete3:faces,ete3:AttrFace,ete3:CircleFace,ete3:TextFace",
       "--keep-going")
show_ca("computational/figures/bca_plddt_vs_tree_dist_and_pid.pdf")

## 4e — stopped-flow kinetics

In [ ]:
run_nb("in_vitro/stopped_flow_analysis.ipynb")
show_ca("in_vitro/endpoints_figure.pdf", "in_vitro/wt_traces.pdf",
        "in_vitro/d152n_traces.pdf", "in_vitro/1360_traces.pdf")

## 4f — natural kcat distribution across carbonic anhydrases

In [ ]:
run_nb("computational/Brenda_CA.ipynb")
show_ca("computational/Brenda_CA.pdf")

## 4a, 4b, 4d — what these still need

**4a, 4b** render the tree through `ete3`'s Qt backend: `pip install PyQt5` and a display
(or `xvfb-run`). **4d** imports a `growth` curve-fitting module that is not on PyPI under
that name and is not vendored in the repository, so the cell raises `ModuleNotFoundError`.
Both are packaging gaps, not analysis gaps.

In [ ]:
# run_nb("computational/tree_analysis.ipynb")   # after: pip install PyQt5
# run_nb("in_vivo/Experimental.ipynb")         # after: the growth parser is packaged